# S6E6 Stellar Class - Max Score Modeling v2

This notebook is designed for Kaggle GPU execution and builds on:

- Your previous OOF result: tuned ensemble `0.9663626`.
- Local top notebooks:
  - `0.96751`: flux transforms, magnitude slope/curvature, Cat/LGB/XGB blend, Nelder-Mead multipliers.
  - `0.96738`: fold-safe target encoding, XGBoost with aggressive class weighting, threshold tuning.
  - `0.96701` / `0.96626`: RealMLP-style neural diversity for blending.
- EDA findings: redshift dominates, color indices matter, categorical signals are strong, train/test shift is negligible.

Main upgrades versus v1:

- More feature families: flux, slope, curvature, redshift-normalized colors, bins, categorical crosses.
- Optional original SDSS17 augmentation if you add `Stellar Classification Dataset - SDSS17` to the Kaggle notebook.
- Multiple seed/model bags for CatBoost, LightGBM, XGBoost, and a PyTorch MLP.
- Fold-safe multiclass target encoding variants.
- Optional loading of previous OOF/test predictions from Kaggle inputs for stacking.
- Weighted blend search, logistic stacking, and class multiplier optimization for balanced accuracy.

Output:

- `submission_max_score_v2.csv`
- per-model submissions
- `oof_diagnostics_v2.csv`
- `max_score_v2_metadata.json`

In [ ]:
import gc
import json
import math
import os
import random
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TARGET = "class"
ID_COL = "id"
CLASSES_ORDER = ["GALAXY", "QSO", "STAR"]
N_CLASSES = 3
N_SPLITS = 5

# For a serious Kaggle run keep these True. For a smoke test, switch heavy options off.
RUN_HEAVY_MODELS = True
RUN_SECOND_SEED_BAGS = True
RUN_TARGET_ENCODING_MODELS = True
RUN_NEURAL_NET = True
USE_GPU = True

# Add the original SDSS17 Kaggle dataset as input, then keep this True.
USE_ORIGINAL_SDSS17 = True
ORIGINAL_WEIGHT = 0.30
ORIGINAL_SDSS17_PATH = Path("/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv")

EARLY_STOPPING_ROUNDS = 250
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

def find_playground_dir():
    candidates = [
        Path("/kaggle/input/playground-series-s6e6"),
        Path("/kaggle/input/competitions/playground-series-s6e6"),
        Path("../input/playground-series-s6e6"),
        Path("../input/competitions/playground-series-s6e6"),
        Path("../data"),
        Path("data"),
    ]
    for p in candidates:
        if (p / "train.csv").exists() and (p / "test.csv").exists():
            return p
    raise FileNotFoundError("Could not find S6E6 train.csv/test.csv")

DATA_DIR = find_playground_dir()
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

## 1. Load Playground Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
le.fit(CLASSES_ORDER)
y = le.transform(train[TARGET])
class_names = le.classes_.tolist()
class_weight_arr = compute_class_weight(class_weight="balanced", classes=np.arange(N_CLASSES), y=y)
class_weight_dict = {i: float(w) for i, w in enumerate(class_weight_arr)}
class_weight_name = {class_names[i]: float(class_weight_arr[i]) for i in range(N_CLASSES)}

print("train:", train.shape, "test:", test.shape)
print("class mapping:", dict(zip(class_names, le.transform(class_names))))
print("class weights:", class_weight_name)
display(train[TARGET].value_counts(normalize=True).rename("share").to_frame())

## 2. Optional Original SDSS17 Loader

In [ ]:
def locate_original_sdss17_csv():
    if ORIGINAL_SDSS17_PATH.exists():
        return ORIGINAL_SDSS17_PATH
    if not Path("/kaggle/input").exists():
        return None
    required = {"alpha", "delta", "u", "g", "r", "i", "z", "redshift", "class"}
    for csv_path in Path("/kaggle/input").rglob("*.csv"):
        if "playground-series-s6e6" in str(csv_path):
            continue
        try:
            head = pd.read_csv(csv_path, nrows=5)
        except Exception:
            continue
        lower_cols = {c.lower() for c in head.columns}
        if required.issubset(lower_cols):
            return csv_path
    return None

def load_original_sdss17():
    path = locate_original_sdss17_csv()
    if path is None:
        print("Original SDSS17 dataset not found. Add it as a Kaggle input to enable augmentation.")
        print("Expected path:", ORIGINAL_SDSS17_PATH)
        return None

    print("Found original SDSS17:", path)
    orig = pd.read_csv(path)
    print("Original raw shape:", orig.shape)
    print("Original columns:", orig.columns.tolist())
    ignored_original_only = [
        c for c in orig.columns
        if c not in ["alpha", "delta", "u", "g", "r", "i", "z", "redshift", "class"]
    ]
    if ignored_original_only:
        print("Ignoring original-only columns unavailable in S6E6 test:", ignored_original_only)
    orig.columns = [c.lower() if c.lower() in {"alpha", "delta", "u", "g", "r", "i", "z", "redshift", "class"} else c for c in orig.columns]
    needed = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift", "class"]
    orig = orig[needed].copy()
    orig[TARGET] = orig["class"].astype(str).str.upper().str.strip()
    orig = orig[orig[TARGET].isin(CLASSES_ORDER)].copy()
    orig = orig.drop_duplicates(subset=["alpha", "delta", "u", "g", "r", "i", "z", "redshift", TARGET]).reset_index(drop=True)

    # Playground-only categorical features do not exist in SDSS17.
    orig["spectral_type"] = "__ORIGINAL_UNKNOWN__"
    orig["galaxy_population"] = "__ORIGINAL_UNKNOWN__"
    orig[ID_COL] = -1 - np.arange(len(orig))
    print("Original rows usable:", orig.shape)
    display(orig[TARGET].value_counts(normalize=True).rename("original_share").to_frame())
    return orig[[ID_COL, "alpha", "delta", "u", "g", "r", "i", "z", "redshift", "spectral_type", "galaxy_population", TARGET]]

original_train = load_original_sdss17() if USE_ORIGINAL_SDSS17 else None

## 3. Feature Engineering

In [ ]:
def reduce_memory(df):
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].astype("float32")
        elif pd.api.types.is_integer_dtype(df[col]) and col != ID_COL:
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

def add_features(df, is_original=False):
    df = df.copy()
    eps = 1e-6
    bands = ["u", "g", "r", "i", "z"]

    for col in ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]:
        df[col] = df[col].astype("float32")
    for col in ["spectral_type", "galaxy_population"]:
        if col not in df.columns:
            df[col] = "__MISSING__"
        df[col] = df[col].astype(str).fillna("__MISSING__")

    df["is_original_sdss17"] = np.int8(is_original)

    # Top-notebook color indices.
    diff_pairs = [
        ("u", "g"), ("g", "r"), ("r", "i"), ("i", "z"),
        ("u", "r"), ("u", "i"), ("u", "z"), ("g", "i"), ("g", "z"), ("r", "z")
    ]
    for a, b in diff_pairs:
        df[f"{a}_{b}"] = df[a] - df[b]
        df[f"{a}_div_{b}"] = df[a] / (df[b].abs() + eps)

    # Magnitude profile.
    df["mag_mean"] = df[bands].mean(axis=1)
    df["mag_std"] = df[bands].std(axis=1)
    df["mag_min"] = df[bands].min(axis=1)
    df["mag_max"] = df[bands].max(axis=1)
    df["mag_range"] = df["mag_max"] - df["mag_min"]
    df["blue_mean"] = df[["u", "g"]].mean(axis=1)
    df["red_mean"] = df[["i", "z"]].mean(axis=1)
    df["blue_minus_red"] = df["blue_mean"] - df["red_mean"]

    # Flux features from 0.96751.
    for band in bands:
        df[f"flux_{band}"] = np.power(10.0, -0.4 * df[band])
    flux_cols = [f"flux_{b}" for b in bands]
    df["flux_mean"] = df[flux_cols].mean(axis=1)
    df["flux_std"] = df[flux_cols].std(axis=1)
    df["flux_min"] = df[flux_cols].min(axis=1)
    df["flux_max"] = df[flux_cols].max(axis=1)
    df["flux_range"] = df["flux_max"] - df["flux_min"]
    for a, b in [("u", "g"), ("g", "r"), ("r", "i"), ("i", "z"), ("u", "z")]:
        df[f"flux_{a}_div_{b}"] = df[f"flux_{a}"] / (df[f"flux_{b}"].abs() + eps)

    x = np.arange(len(bands), dtype=np.float32)
    x_centered = x - x.mean()
    denom = float(np.sum(x_centered ** 2))
    df["mag_slope"] = df[bands].sub(df[bands].mean(axis=1), axis=0).dot(x_centered) / denom
    df["mag_curvature"] = df["u"] - 2 * df["r"] + df["z"]
    df["blue_curvature"] = df["u"] - 2 * df["g"] + df["r"]
    df["red_curvature"] = df["r"] - 2 * df["i"] + df["z"]

    # Redshift is the dominant EDA feature.
    df["redshift_abs"] = df["redshift"].abs()
    df["redshift_sq"] = df["redshift"] ** 2
    df["redshift_cu"] = df["redshift"] ** 3
    df["redshift_log1p"] = np.log1p(df["redshift"].clip(lower=0))
    df["near_zero_redshift"] = (df["redshift_abs"] < 0.10).astype("int8")
    df["high_redshift"] = (df["redshift"] > 1.0).astype("int8")
    df["very_high_redshift"] = (df["redshift"] > 2.0).astype("int8")
    for col in bands + ["mag_mean", "mag_std", "mag_slope", "mag_curvature", "blue_minus_red", "u_g", "g_r", "r_i", "i_z"]:
        df[f"redshift_x_{col}"] = df["redshift"] * df[col]
        df[f"{col}_per_redshift"] = df[col] / (df["redshift_abs"] + eps)

    # Sky features.
    alpha_rad = np.deg2rad(df["alpha"])
    delta_rad = np.deg2rad(df["delta"])
    df["alpha_sin"] = np.sin(alpha_rad)
    df["alpha_cos"] = np.cos(alpha_rad)
    df["delta_sin"] = np.sin(delta_rad)
    df["delta_cos"] = np.cos(delta_rad)
    df["sky_x"] = np.cos(delta_rad) * np.cos(alpha_rad)
    df["sky_y"] = np.cos(delta_rad) * np.sin(alpha_rad)
    df["sky_z"] = np.sin(delta_rad)
    df["alpha_delta_interaction"] = df["alpha"] * df["delta"]

    # Ordinal and cross categorical signals.
    spectral_ord = {"O/B": 0.0, "A/F": 1.5, "G/K": 3.5, "M": 5.0, "__ORIGINAL_UNKNOWN__": -1.0}
    df["spectral_ord"] = df["spectral_type"].map(spectral_ord).fillna(-1).astype("float32")
    df["spectral_population"] = df["spectral_type"] + "__" + df["galaxy_population"]
    df["spectral_highz"] = df["spectral_type"] + "__highz_" + df["high_redshift"].astype(str)
    df["population_nearzeroz"] = df["galaxy_population"] + "__nearz_" + df["near_zero_redshift"].astype(str)

    # Fold-safe target encoding candidates. These are categorical bins/crosses, not direct label features.
    df["redshift_bin"] = np.floor(np.clip(df["redshift"], -0.5, 4.5) * 10).astype("int16").astype(str)
    df["mag_mean_bin"] = np.floor(np.clip(df["mag_mean"], 10, 35)).astype("int16").astype(str)
    df["u_g_bin"] = np.floor(np.clip(df["u_g"], -5, 8) * 2).astype("int16").astype(str)
    df["spectral_redshift_bin"] = df["spectral_type"] + "__rz_" + df["redshift_bin"]
    df["population_redshift_bin"] = df["galaxy_population"] + "__rz_" + df["redshift_bin"]
    df["spectral_mag_bin"] = df["spectral_type"] + "__mag_" + df["mag_mean_bin"]
    df["population_color_bin"] = df["galaxy_population"] + "__ug_" + df["u_g_bin"]

    return reduce_memory(df)

train_base = train.drop(columns=[TARGET]).copy()
test_base = test.copy()
train_fe = add_features(train_base, is_original=False)
test_fe = add_features(test_base, is_original=False)

if original_train is not None and len(original_train):
    orig_y = le.transform(original_train[TARGET])
    orig_fe = add_features(original_train.drop(columns=[TARGET]), is_original=True)
else:
    orig_y = None
    orig_fe = None

feature_cols = [c for c in train_fe.columns if c != ID_COL]
cat_cols = train_fe[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]
te_cols = [
    "spectral_population", "spectral_highz", "population_nearzeroz",
    "redshift_bin", "mag_mean_bin", "u_g_bin",
    "spectral_redshift_bin", "population_redshift_bin", "spectral_mag_bin", "population_color_bin"
]
te_cols = [c for c in te_cols if c in feature_cols]

for df in [train_fe, test_fe, orig_fe]:
    if df is None:
        continue
    for c in cat_cols:
        df[c] = df[c].astype("category")

print("feature count:", len(feature_cols), "numeric:", len(num_cols), "categorical:", len(cat_cols), "TE:", len(te_cols))
print("categorical:", cat_cols)
print("target encoding columns:", te_cols)
if orig_fe is not None:
    print("original augmented rows:", orig_fe.shape)
display(train_fe[feature_cols].head())

## 4. CV, Scoring, Encoding, And Previous Prediction Loading

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_id = np.zeros(len(train_fe), dtype=np.int8)
for fold, (_, valid_idx) in enumerate(skf.split(train_fe, y)):
    fold_id[valid_idx] = fold
display(pd.crosstab(fold_id, train[TARGET], normalize="index"))

def sample_weights_for(y_values, original_flags=None, star_boost=1.0):
    w = np.asarray([class_weight_dict[int(v)] for v in y_values], dtype=np.float32)
    # Optional extra pressure on STAR recall; useful because balanced accuracy values recall equally.
    star_idx = int(le.transform(["STAR"])[0])
    w[np.asarray(y_values) == star_idx] *= star_boost
    if original_flags is not None:
        w = w * np.where(np.asarray(original_flags) == 1, ORIGINAL_WEIGHT, 1.0).astype(np.float32)
    return w

def balanced_acc(y_true, proba, multipliers=None):
    p = proba if multipliers is None else proba * np.asarray(multipliers)[None, :]
    return balanced_accuracy_score(y_true, p.argmax(axis=1))

def show_score(name, y_true, proba, multipliers=None):
    p = proba if multipliers is None else proba * np.asarray(multipliers)[None, :]
    pred = p.argmax(axis=1)
    score = balanced_accuracy_score(y_true, pred)
    print(f"\n{name}: balanced_accuracy={score:.8f}")
    print(classification_report(y_true, pred, target_names=class_names, digits=5))
    display(pd.DataFrame(confusion_matrix(y_true, pred), index=class_names, columns=class_names))
    return score

def label_encode_splits(X_trn, X_val, X_test, cols):
    X_trn = X_trn.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()
    for col in cols:
        cats = pd.Index(X_trn[col].astype(str).unique()).union(pd.Index(X_val[col].astype(str).unique())).union(pd.Index(X_test[col].astype(str).unique()))
        mapping = {v: i for i, v in enumerate(cats)}
        X_trn[col] = X_trn[col].astype(str).map(mapping).fillna(-1).astype("int32")
        X_val[col] = X_val[col].astype(str).map(mapping).fillna(-1).astype("int32")
        X_test[col] = X_test[col].astype(str).map(mapping).fillna(-1).astype("int32")
    return X_trn, X_val, X_test

def _target_encode_apply(train_col, y_train, apply_col, prior, smooth=20.0):
    tmp = pd.DataFrame({"key": train_col.astype(str), "y": y_train})
    counts = tmp.groupby("key")["y"].size().astype("float32")
    encoded_parts = []
    for cls_idx in range(N_CLASSES):
        cls_sum = tmp.assign(v=(tmp["y"].values == cls_idx).astype("float32")).groupby("key")["v"].sum()
        mean = (cls_sum + prior[cls_idx] * smooth) / (counts + smooth)
        encoded_parts.append(apply_col.astype(str).map(mean).fillna(prior[cls_idx]).astype("float32").values)
    return np.vstack(encoded_parts).T

def apply_fold_target_encoding(X_trn, X_val, X_test, y_trn, cols):
    if not cols:
        return X_trn, X_val, X_test
    y_trn = np.asarray(y_trn)
    prior = np.bincount(y_trn, minlength=N_CLASSES).astype("float32")
    prior = prior / prior.sum()
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_blocks, val_blocks, test_blocks, names = [], [], [], []
    for col in cols:
        tr_encoded = np.zeros((len(X_trn), N_CLASSES), dtype="float32")
        col_values = X_trn[col].astype(str).reset_index(drop=True)
        for inner_tr_idx, inner_va_idx in inner.split(np.zeros(len(X_trn)), y_trn):
            tr_encoded[inner_va_idx] = _target_encode_apply(
                col_values.iloc[inner_tr_idx],
                y_trn[inner_tr_idx],
                col_values.iloc[inner_va_idx],
                prior,
            )
        val_encoded = _target_encode_apply(X_trn[col], y_trn, X_val[col], prior)
        test_encoded = _target_encode_apply(X_trn[col], y_trn, X_test[col], prior)
        train_blocks.append(tr_encoded)
        val_blocks.append(val_encoded)
        test_blocks.append(test_encoded)
        names.extend([f"te_{col}_{cls}" for cls in class_names])
    tr = np.hstack(train_blocks)
    va = np.hstack(val_blocks)
    te = np.hstack(test_blocks)
    tr_df = pd.DataFrame(tr, columns=names, index=X_trn.index)
    va_df = pd.DataFrame(va, columns=names, index=X_val.index)
    te_df = pd.DataFrame(te, columns=names, index=X_test.index)
    return (
        pd.concat([X_trn.drop(columns=cols), tr_df], axis=1),
        pd.concat([X_val.drop(columns=cols), va_df], axis=1),
        pd.concat([X_test.drop(columns=cols), te_df], axis=1),
    )

def make_fold_train_valid(fold, use_original=True):
    trn_idx = np.where(fold_id != fold)[0]
    val_idx = np.where(fold_id == fold)[0]
    X_trn = train_fe.iloc[trn_idx][feature_cols].copy()
    y_trn = y[trn_idx]
    orig_flags = np.zeros(len(X_trn), dtype=np.int8)
    if use_original and orig_fe is not None and orig_y is not None:
        X_trn = pd.concat([X_trn, orig_fe[feature_cols]], axis=0, ignore_index=True)
        y_trn = np.concatenate([y_trn, orig_y])
        orig_flags = np.concatenate([orig_flags, np.ones(len(orig_fe), dtype=np.int8)])
    X_val = train_fe.iloc[val_idx][feature_cols].copy()
    y_val = y[val_idx]
    X_test = test_fe[feature_cols].copy()
    return X_trn, y_trn, orig_flags, X_val, y_val, val_idx, X_test

def align_category_dtype(X_trn, X_val, X_test, cols):
    for c in cols:
        cats = pd.Index(X_trn[c].astype(str).unique()).union(pd.Index(X_val[c].astype(str).unique())).union(pd.Index(X_test[c].astype(str).unique()))
        X_trn[c] = pd.Categorical(X_trn[c].astype(str), categories=cats)
        X_val[c] = pd.Categorical(X_val[c].astype(str), categories=cats)
        X_test[c] = pd.Categorical(X_test[c].astype(str), categories=cats)
    return X_trn, X_val, X_test

def save_submission(name, test_proba, multipliers=None):
    p = test_proba if multipliers is None else test_proba * np.asarray(multipliers)[None, :]
    labels = le.inverse_transform(p.argmax(axis=1))
    sub = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET: labels})
    path = OUT_DIR / name
    sub.to_csv(path, index=False)
    print("saved:", path, sub.shape)
    display(sub.head())
    return sub

oof_predictions = {}
test_predictions = {}
model_scores = {}

def maybe_load_prediction_file(path):
    try:
        if path.suffix == ".npy":
            arr = np.load(path)
        elif path.suffix in [".csv", ".txt"]:
            df = pd.read_csv(path)
            use_cols = [c for c in df.columns if str(c).upper() in CLASSES_ORDER or str(c).lower().startswith("pred") or str(c).isdigit()]
            if len(use_cols) >= 3:
                arr = df[use_cols[:3]].values
            elif df.shape[1] == 3:
                arr = df.values
            else:
                return None
        elif path.suffix == ".parquet":
            arr = pd.read_parquet(path).values
        else:
            return None
        arr = np.asarray(arr, dtype=np.float32)
        if arr.ndim == 2 and arr.shape[1] == N_CLASSES:
            return arr
    except Exception:
        return None
    return None

def load_previous_predictions():
    # Add previous notebook outputs as Kaggle datasets and this block will blend them automatically.
    if not Path("/kaggle/input").exists():
        return
    files = []
    for p in Path("/kaggle/input").rglob("*"):
        if p.suffix.lower() in [".npy", ".csv", ".parquet", ".txt"]:
            name = p.name.lower()
            if any(k in name for k in ["oof", "test_pred", "mdl_preds", "model_pred", "preds"]):
                files.append(p)
    for p in files:
        arr = maybe_load_prediction_file(p)
        if arr is None:
            continue
        key = re.sub(r"[^a-zA-Z0-9_]+", "_", p.stem)[:45]
        if arr.shape[0] == len(train_fe):
            oof_predictions[f"external_oof_{key}"] = arr
            print("loaded external OOF:", p, arr.shape)
        elif arr.shape[0] == len(test_fe):
            test_predictions[f"external_test_{key}"] = arr
            print("loaded external TEST:", p, arr.shape)

load_previous_predictions()

## 5. CatBoost Seed Bags

In [ ]:
try:
    from catboost import CatBoostClassifier, Pool
    HAS_CATBOOST = True
except Exception as e:
    HAS_CATBOOST = False
    print("CatBoost unavailable:", repr(e))

def train_catboost(name, seed=42, depth=10, lr=0.03, star_boost=1.0, use_original=True):
    cat_oof = np.zeros((len(train_fe), N_CLASSES), dtype=np.float32)
    cat_test = np.zeros((len(test_fe), N_CLASSES), dtype=np.float32)
    cat_idx = [feature_cols.index(c) for c in cat_cols]

    params = dict(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=6500 if RUN_HEAVY_MODELS else 1600,
        learning_rate=lr,
        depth=depth,
        l2_leaf_reg=5.0,
        random_strength=0.7,
        bootstrap_type="Bayesian",
        bagging_temperature=0.8,
        border_count=254,
        random_seed=seed,
        allow_writing_files=False,
        verbose=500,
    )
    if USE_GPU:
        params.update(task_type="GPU", devices="0")

    for fold in range(N_SPLITS):
        print(f"\n{name} fold {fold}")
        X_trn, y_trn, orig_flags, X_val, y_val, val_idx, X_test = make_fold_train_valid(fold, use_original=use_original)
        sw = sample_weights_for(y_trn, orig_flags, star_boost=star_boost)
        train_pool = Pool(X_trn, y_trn, cat_features=cat_idx, weight=sw)
        val_pool = Pool(X_val, y_val, cat_features=cat_idx)
        test_pool = Pool(X_test, cat_features=cat_idx)
        model = CatBoostClassifier(**params)
        try:
            model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True)
        except Exception as gpu_error:
            print("CatBoost GPU failed; retry CPU:", repr(gpu_error))
            cpu_params = params.copy()
            cpu_params.pop("task_type", None)
            cpu_params.pop("devices", None)
            cpu_params["thread_count"] = -1
            model = CatBoostClassifier(**cpu_params)
            model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True)
        cat_oof[val_idx] = model.predict_proba(val_pool).astype(np.float32)
        cat_test += model.predict_proba(test_pool).astype(np.float32) / N_SPLITS
        print("fold score:", balanced_acc(y_val, cat_oof[val_idx]))
        del model, train_pool, val_pool, test_pool, X_trn, X_val, X_test
        gc.collect()
    oof_predictions[name] = cat_oof
    test_predictions[name] = cat_test
    model_scores[name] = show_score(name, y, cat_oof)
    save_submission(f"submission_{name}.csv", cat_test)

if HAS_CATBOOST:
    train_catboost("cat_d10_s42_aug", seed=42, depth=10, lr=0.030, star_boost=1.00, use_original=True)
    if RUN_SECOND_SEED_BAGS:
        train_catboost("cat_d8_s2026_aug", seed=2026, depth=8, lr=0.035, star_boost=1.05, use_original=True)

## 6. LightGBM Seed Bags And Target Encoding Variant

In [ ]:
try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception as e:
    HAS_LIGHTGBM = False
    print("LightGBM unavailable:", repr(e))

def train_lightgbm(name, seed=42, use_te=False, star_boost=1.0, num_leaves=160, lr=0.025, use_original=True):
    lgb_oof = np.zeros((len(train_fe), N_CLASSES), dtype=np.float32)
    lgb_test = np.zeros((len(test_fe), N_CLASSES), dtype=np.float32)
    params = dict(
        objective="multiclass",
        num_class=N_CLASSES,
        n_estimators=8500 if RUN_HEAVY_MODELS else 1800,
        learning_rate=lr,
        num_leaves=num_leaves,
        min_child_samples=55,
        subsample=0.88,
        subsample_freq=1,
        colsample_bytree=0.84,
        reg_alpha=0.06,
        reg_lambda=1.4,
        max_bin=255,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )
    if USE_GPU:
        params.update(device_type="gpu", gpu_use_dp=False)

    for fold in range(N_SPLITS):
        print(f"\n{name} fold {fold}")
        X_trn, y_trn, orig_flags, X_val, y_val, val_idx, X_test = make_fold_train_valid(fold, use_original=use_original)
        active_cat_cols = cat_cols.copy()
        if use_te:
            X_trn, X_val, X_test = apply_fold_target_encoding(X_trn, X_val, X_test, y_trn, te_cols)
            active_cat_cols = [c for c in active_cat_cols if c not in te_cols]
        X_trn, X_val, X_test = align_category_dtype(X_trn, X_val, X_test, active_cat_cols)
        sw = sample_weights_for(y_trn, orig_flags, star_boost=star_boost)
        model = lgb.LGBMClassifier(**params)
        try:
            model.fit(
                X_trn, y_trn,
                sample_weight=sw,
                eval_set=[(X_val, y_val)],
                eval_metric="multi_logloss",
                categorical_feature=active_cat_cols,
                callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(500)],
            )
        except Exception as gpu_error:
            print("LightGBM GPU failed; retry CPU:", repr(gpu_error))
            cpu_params = params.copy()
            cpu_params.pop("device_type", None)
            cpu_params.pop("gpu_use_dp", None)
            model = lgb.LGBMClassifier(**cpu_params)
            model.fit(
                X_trn, y_trn,
                sample_weight=sw,
                eval_set=[(X_val, y_val)],
                eval_metric="multi_logloss",
                categorical_feature=active_cat_cols,
                callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(500)],
            )
        lgb_oof[val_idx] = model.predict_proba(X_val).astype(np.float32)
        lgb_test += model.predict_proba(X_test).astype(np.float32) / N_SPLITS
        print("fold score:", balanced_acc(y_val, lgb_oof[val_idx]))
        del model, X_trn, X_val, X_test
        gc.collect()
    oof_predictions[name] = lgb_oof
    test_predictions[name] = lgb_test
    model_scores[name] = show_score(name, y, lgb_oof)
    save_submission(f"submission_{name}.csv", lgb_test)

if HAS_LIGHTGBM:
    train_lightgbm("lgb_s42_aug", seed=42, use_te=False, star_boost=1.00, num_leaves=160, lr=0.025, use_original=True)
    if RUN_TARGET_ENCODING_MODELS:
        train_lightgbm("lgb_te_s777_aug", seed=777, use_te=True, star_boost=1.03, num_leaves=128, lr=0.028, use_original=True)

## 7. XGBoost Seed Bags And Target Encoding Variant

In [ ]:
try:
    from xgboost import XGBClassifier
    import xgboost as xgb
    HAS_XGBOOST = True
    print("XGBoost:", xgb.__version__)
except Exception as e:
    HAS_XGBOOST = False
    print("XGBoost unavailable:", repr(e))

def train_xgboost(name, seed=42, use_te=False, star_boost=1.0, max_depth=8, lr=0.015, use_original=True):
    xgb_oof = np.zeros((len(train_fe), N_CLASSES), dtype=np.float32)
    xgb_test = np.zeros((len(test_fe), N_CLASSES), dtype=np.float32)
    params = dict(
        objective="multi:softprob",
        num_class=N_CLASSES,
        eval_metric="mlogloss",
        tree_method="hist",
        n_estimators=7000 if RUN_HEAVY_MODELS else 1800,
        learning_rate=lr,
        max_depth=max_depth,
        min_child_weight=5,
        max_delta_step=1,
        gamma=0.20,
        reg_alpha=0.45,
        reg_lambda=2.5,
        subsample=0.76,
        colsample_bytree=0.72,
        colsample_bylevel=0.82,
        max_bin=512,
        random_state=seed,
        n_jobs=-1,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    if USE_GPU:
        params.update(device="cuda")

    for fold in range(N_SPLITS):
        print(f"\n{name} fold {fold}")
        X_trn, y_trn, orig_flags, X_val, y_val, val_idx, X_test = make_fold_train_valid(fold, use_original=use_original)
        if use_te:
            X_trn, X_val, X_test = apply_fold_target_encoding(X_trn, X_val, X_test, y_trn, te_cols)
        active_cat_cols = [c for c in cat_cols if c in X_trn.columns]
        X_trn, X_val, X_test = label_encode_splits(X_trn, X_val, X_test, active_cat_cols)
        sw = sample_weights_for(y_trn, orig_flags, star_boost=star_boost)
        model = XGBClassifier(**params)
        try:
            model.fit(X_trn, y_trn, sample_weight=sw, eval_set=[(X_val, y_val)], verbose=500)
        except Exception as gpu_error:
            print("XGBoost GPU failed; retry CPU:", repr(gpu_error))
            cpu_params = params.copy()
            cpu_params.pop("device", None)
            model = XGBClassifier(**cpu_params)
            model.fit(X_trn, y_trn, sample_weight=sw, eval_set=[(X_val, y_val)], verbose=500)
        xgb_oof[val_idx] = model.predict_proba(X_val).astype(np.float32)
        xgb_test += model.predict_proba(X_test).astype(np.float32) / N_SPLITS
        print("fold score:", balanced_acc(y_val, xgb_oof[val_idx]))
        del model, X_trn, X_val, X_test
        gc.collect()
    oof_predictions[name] = xgb_oof
    test_predictions[name] = xgb_test
    model_scores[name] = show_score(name, y, xgb_oof)
    save_submission(f"submission_{name}.csv", xgb_test)

if HAS_XGBOOST:
    train_xgboost("xgb_top_s42_aug", seed=42, use_te=False, star_boost=1.00, max_depth=8, lr=0.015, use_original=True)
    if RUN_TARGET_ENCODING_MODELS:
        train_xgboost("xgb_te_s2026_aug", seed=2026, use_te=True, star_boost=1.08, max_depth=7, lr=0.020, use_original=True)

## 8. RealMLP-Inspired PyTorch Model

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print("PyTorch unavailable:", repr(e))

class TabMLP(nn.Module):
    def __init__(self, n_num, cat_cardinalities, n_classes=3, emb_dim=8):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(int(card) + 1, min(emb_dim, max(2, int(round(card ** 0.25)) + 2)))
            for card in cat_cardinalities
        ])
        emb_out = sum(e.embedding_dim for e in self.embeddings)
        self.net = nn.Sequential(
            nn.BatchNorm1d(n_num + emb_out),
            nn.Linear(n_num + emb_out, 512),
            nn.SiLU(),
            nn.Dropout(0.08),
            nn.Linear(512, 384),
            nn.SiLU(),
            nn.Dropout(0.06),
            nn.Linear(384, 192),
            nn.SiLU(),
            nn.Dropout(0.04),
            nn.Linear(192, n_classes),
        )
    def forward(self, x_num, x_cat):
        if len(self.embeddings):
            emb = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embeddings)]
            x = torch.cat([x_num] + emb, dim=1)
        else:
            x = x_num
        return self.net(x)

def fit_transform_mlp_tables(X_trn, X_val, X_test):
    X_trn = X_trn.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()
    scaler = RobustScaler()
    Xn_trn = scaler.fit_transform(X_trn[num_cols]).astype("float32")
    Xn_val = scaler.transform(X_val[num_cols]).astype("float32")
    Xn_test = scaler.transform(X_test[num_cols]).astype("float32")
    cat_cards = []
    cat_arrays = []
    for c in cat_cols:
        cats = pd.Index(X_trn[c].astype(str).unique()).union(pd.Index(X_val[c].astype(str).unique())).union(pd.Index(X_test[c].astype(str).unique()))
        mapping = {v: i + 1 for i, v in enumerate(cats)}
        cat_cards.append(len(mapping) + 1)
        cat_arrays.append((
            X_trn[c].astype(str).map(mapping).fillna(0).astype("int64").values,
            X_val[c].astype(str).map(mapping).fillna(0).astype("int64").values,
            X_test[c].astype(str).map(mapping).fillna(0).astype("int64").values,
        ))
    Xc_trn = np.vstack([a[0] for a in cat_arrays]).T if cat_arrays else np.zeros((len(X_trn), 0), dtype="int64")
    Xc_val = np.vstack([a[1] for a in cat_arrays]).T if cat_arrays else np.zeros((len(X_val), 0), dtype="int64")
    Xc_test = np.vstack([a[2] for a in cat_arrays]).T if cat_arrays else np.zeros((len(X_test), 0), dtype="int64")
    return Xn_trn, Xc_trn, Xn_val, Xc_val, Xn_test, Xc_test, cat_cards

def train_mlp(name="torch_mlp_s42_aug", seed=42, use_original=True):
    if not HAS_TORCH or not RUN_NEURAL_NET:
        return
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch device:", device)
    torch.manual_seed(seed)
    mlp_oof = np.zeros((len(train_fe), N_CLASSES), dtype=np.float32)
    mlp_test = np.zeros((len(test_fe), N_CLASSES), dtype=np.float32)

    for fold in range(N_SPLITS):
        print(f"\n{name} fold {fold}")
        X_trn, y_trn, orig_flags, X_val, y_val, val_idx, X_test = make_fold_train_valid(fold, use_original=use_original)
        Xn_trn, Xc_trn, Xn_val, Xc_val, Xn_test, Xc_test, cat_cards = fit_transform_mlp_tables(X_trn, X_val, X_test)
        sw = sample_weights_for(y_trn, orig_flags, star_boost=1.03)

        train_ds = TensorDataset(
            torch.tensor(Xn_trn), torch.tensor(Xc_trn), torch.tensor(y_trn.astype("int64")), torch.tensor(sw.astype("float32"))
        )
        train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True, num_workers=2, pin_memory=True)
        model = TabMLP(Xn_trn.shape[1], cat_cards, N_CLASSES, emb_dim=8).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2.0e-3, weight_decay=1.2e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=14 if RUN_HEAVY_MODELS else 5)
        ce = nn.CrossEntropyLoss(reduction="none", label_smoothing=0.035)
        best_score = -1
        best_state = None
        epochs = 14 if RUN_HEAVY_MODELS else 5
        Xn_val_t = torch.tensor(Xn_val).to(device)
        Xc_val_t = torch.tensor(Xc_val).to(device)

        for epoch in range(epochs):
            model.train()
            total_loss = 0.0
            for xb_num, xb_cat, yb, wb in train_loader:
                xb_num = xb_num.to(device, non_blocking=True)
                xb_cat = xb_cat.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                wb = wb.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                logits = model(xb_num, xb_cat)
                loss = (ce(logits, yb) * wb).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += float(loss.detach().cpu())
            scheduler.step()
            model.eval()
            with torch.no_grad():
                val_logits = model(Xn_val_t, Xc_val_t)
                val_proba = torch.softmax(val_logits, dim=1).detach().cpu().numpy()
            score = balanced_acc(y_val, val_proba)
            print(f"epoch {epoch+1}/{epochs} loss={total_loss/len(train_loader):.5f} val_bal={score:.6f}")
            if score > best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            mlp_oof[val_idx] = torch.softmax(model(Xn_val_t, Xc_val_t), dim=1).detach().cpu().numpy()
            test_parts = []
            for start in range(0, len(Xn_test), 65536):
                xb_num = torch.tensor(Xn_test[start:start+65536]).to(device)
                xb_cat = torch.tensor(Xc_test[start:start+65536]).to(device)
                test_parts.append(torch.softmax(model(xb_num, xb_cat), dim=1).detach().cpu().numpy())
            mlp_test += np.vstack(test_parts).astype(np.float32) / N_SPLITS
        print("fold score:", balanced_acc(y_val, mlp_oof[val_idx]))
        del model, train_loader, train_ds, Xn_trn, Xc_trn, Xn_val, Xc_val, Xn_test, Xc_test
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    oof_predictions[name] = mlp_oof
    test_predictions[name] = mlp_test
    model_scores[name] = show_score(name, y, mlp_oof)
    save_submission(f"submission_{name}.csv", mlp_test)

train_mlp()

## 9. Blend, Stack, And Class Multiplier Optimization

In [ ]:
# Pair external OOF/test predictions if names do not match exactly.
# Named model predictions from this notebook already match in both dictionaries.
usable_names = [n for n in oof_predictions if n in test_predictions and oof_predictions[n].shape == (len(train_fe), N_CLASSES) and test_predictions[n].shape == (len(test_fe), N_CLASSES)]
external_oofs = [n for n in oof_predictions if n.startswith("external_oof")]
external_tests = [n for n in test_predictions if n.startswith("external_test")]
for oo, tt in zip(sorted(external_oofs), sorted(external_tests)):
    merged_name = "external_pair_" + oo.replace("external_oof_", "")[:25]
    if merged_name not in oof_predictions:
        oof_predictions[merged_name] = oof_predictions[oo]
        test_predictions[merged_name] = test_predictions[tt]
        usable_names.append(merged_name)

usable_names = [n for n in dict.fromkeys(usable_names) if n in oof_predictions and n in test_predictions]
assert len(usable_names) > 0, "No usable model predictions available."
print("Usable models:", usable_names)

for n in usable_names:
    model_scores[n] = show_score(n, y, oof_predictions[n])

def blend_from_weights(names, weights, pred_dict):
    weights = np.asarray(weights, dtype=np.float64)
    weights = np.clip(weights, 0, None)
    weights = weights / weights.sum()
    out = np.zeros_like(pred_dict[names[0]], dtype=np.float64)
    for w, n in zip(weights, names):
        out += w * pred_dict[n]
    return out.astype(np.float32)

def random_blend_search(names, rounds=12000):
    rng = np.random.default_rng(SEED)
    candidates = []
    candidates.append(np.ones(len(names)) / len(names))
    for i in range(len(names)):
        w = np.zeros(len(names)); w[i] = 1.0; candidates.append(w)
    for alpha in [0.2, 0.4, 0.7, 1.0, 2.0, 5.0]:
        n_rounds = rounds // 6
        for _ in range(n_rounds):
            candidates.append(rng.dirichlet(np.ones(len(names)) * alpha))
    best_score, best_w = -1, None
    for w in candidates:
        score = balanced_acc(y, blend_from_weights(names, w, oof_predictions))
        if score > best_score:
            best_score, best_w = score, w
    return np.asarray(best_w) / np.asarray(best_w).sum(), best_score

def optimize_multipliers(proba):
    def obj(log_m):
        m = np.exp(log_m)
        m = m / m.mean()
        return -balanced_acc(y, proba, m)
    best = np.zeros(N_CLASSES)
    best_val = obj(best)
    starts = [
        np.zeros(N_CLASSES),
        np.log(np.array([0.68, 1.01, 1.01])),
        np.log(np.array([0.75, 1.00, 1.10])),
        np.log(np.array([0.80, 1.05, 1.20])),
    ]
    for s in starts:
        res = minimize(obj, s, method="Nelder-Mead", options={"maxiter": 2500, "xatol": 1e-8, "fatol": 1e-8})
        if res.fun < best_val:
            best_val, best = res.fun, res.x
    m = np.exp(best)
    m = m / m.mean()
    return m.astype(np.float32), -best_val

best_w, best_blend_score = random_blend_search(usable_names, rounds=18000 if RUN_HEAVY_MODELS else 3000)
blend_oof = blend_from_weights(usable_names, best_w, oof_predictions)
blend_test = blend_from_weights(usable_names, best_w, test_predictions)
print("Best raw blend:", best_blend_score)
print("Blend weights:", dict(zip(usable_names, [float(x) for x in best_w])))
blend_mult, blend_tuned_score = optimize_multipliers(blend_oof)
show_score("weighted_blend_raw", y, blend_oof)
show_score("weighted_blend_tuned", y, blend_oof, blend_mult)

# Logistic stacking on OOF predictions. This gives a different decision surface from fixed weights.
def make_meta_features(pred_dict, names):
    blocks = []
    for n in names:
        p = np.clip(pred_dict[n], 1e-6, 1 - 1e-6)
        blocks.append(p)
        blocks.append(np.log(p))
    return np.hstack(blocks).astype(np.float32)

meta_X = make_meta_features(oof_predictions, usable_names)
meta_test_X = make_meta_features(test_predictions, usable_names)
stack_oof = np.zeros((len(train_fe), N_CLASSES), dtype=np.float32)
stack_test = np.zeros((len(test_fe), N_CLASSES), dtype=np.float32)
for fold, (tr_idx, va_idx) in enumerate(skf.split(meta_X, y)):
    meta = LogisticRegression(
        C=0.18,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=2000,
        random_state=SEED + fold,
    )
    meta.fit(meta_X[tr_idx], y[tr_idx])
    stack_oof[va_idx] = meta.predict_proba(meta_X[va_idx]).astype(np.float32)
    stack_test += meta.predict_proba(meta_test_X).astype(np.float32) / N_SPLITS
stack_raw_score = show_score("logistic_stack_raw", y, stack_oof)
stack_mult, stack_tuned_score = optimize_multipliers(stack_oof)
show_score("logistic_stack_tuned", y, stack_oof, stack_mult)

final_candidates = {
    "weighted_blend": (blend_oof, blend_test, blend_mult, blend_tuned_score),
    "logistic_stack": (stack_oof, stack_test, stack_mult, stack_tuned_score),
}
best_final_name = max(final_candidates, key=lambda k: final_candidates[k][3])
final_oof, final_test, final_mult, final_score = final_candidates[best_final_name]
print("FINAL choice:", best_final_name, final_score)

save_submission("submission_max_score_v2.csv", final_test, final_mult)
save_submission("submission_weighted_blend_v2.csv", blend_test, blend_mult)
save_submission("submission_logistic_stack_v2.csv", stack_test, stack_mult)

diag = []
for n in usable_names:
    diag.append({"model": n, "balanced_accuracy_raw": balanced_acc(y, oof_predictions[n])})
diag.extend([
    {"model": "weighted_blend_raw", "balanced_accuracy_raw": balanced_acc(y, blend_oof)},
    {"model": "weighted_blend_tuned", "balanced_accuracy_raw": balanced_acc(y, blend_oof, blend_mult)},
    {"model": "logistic_stack_raw", "balanced_accuracy_raw": stack_raw_score},
    {"model": "logistic_stack_tuned", "balanced_accuracy_raw": stack_tuned_score},
    {"model": "FINAL_" + best_final_name, "balanced_accuracy_raw": final_score},
])
diag_df = pd.DataFrame(diag).sort_values("balanced_accuracy_raw", ascending=False)
display(diag_df)
diag_df.to_csv(OUT_DIR / "oof_diagnostics_v2.csv", index=False)

for n in usable_names:
    safe = re.sub(r"[^a-zA-Z0-9_]+", "_", n)
    np.save(OUT_DIR / f"oof_{safe}.npy", oof_predictions[n])
    np.save(OUT_DIR / f"test_preds_{safe}.npy", test_predictions[n])
np.save(OUT_DIR / "final_oof_v2.npy", final_oof)
np.save(OUT_DIR / "final_test_v2.npy", final_test)
with open(OUT_DIR / "max_score_v2_metadata.json", "w") as f:
    json.dump(
        {
            "final_choice": best_final_name,
            "final_oof_balanced_accuracy": float(final_score),
            "usable_models": usable_names,
            "blend_weights": dict(zip(usable_names, [float(x) for x in best_w])),
            "blend_multipliers": dict(zip(class_names, [float(x) for x in blend_mult])),
            "stack_multipliers": dict(zip(class_names, [float(x) for x in stack_mult])),
            "used_original_sdss17": bool(orig_fe is not None),
            "original_weight": ORIGINAL_WEIGHT,
            "feature_count": len(feature_cols),
            "te_cols": te_cols,
        },
        f,
        indent=2,
    )

## 10. Submission Sanity Check

In [ ]:
sub = pd.read_csv(OUT_DIR / "submission_max_score_v2.csv")
print(sub.shape)
display(sub.head())
display(sub[TARGET].value_counts(normalize=True).rename("submission_share").to_frame())
assert sub.shape[0] == len(test)
assert list(sub.columns) == [ID_COL, TARGET]
assert sub[ID_COL].equals(test[ID_COL])
assert set(sub[TARGET]).issubset(set(class_names))
print("Ready:", OUT_DIR / "submission_max_score_v2.csv")

## Kaggle Run Notes

1. Add the competition dataset.
2. Add `Stellar Classification Dataset - SDSS17` as an input if you want original-data augmentation.
3. Optionally add your previous v1 notebook output as a Kaggle dataset; this notebook will auto-load OOF/test prediction arrays when filenames contain `oof`, `preds`, or `Mdl_Preds`.
4. Enable GPU. T4 is enough; P100 is also fine. If LightGBM GPU fails, the notebook retries CPU.
5. For leaderboard probing, compare `submission_max_score_v2.csv`, `submission_weighted_blend_v2.csv`, and `submission_logistic_stack_v2.csv`.